# 01 — Audit cấu hình & Xây dựng Run Manifest

Notebook này thực hiện **Section 3** của kế hoạch phân tích: audit toàn bộ log
`full/` và `no-EQ/` trước khi tính bất kỳ kết quả nào.

Mục tiêu:
1. Quét toàn bộ (instance, seed) của cả hai variant.
2. Parse `_config.txt` và `_summary.txt` để kiểm tra tính nhất quán cấu hình
   (maxIterations, operators, penalty, charging model...).
3. Xây `run_manifest.csv` — nguồn sự thật duy nhất cho các notebook sau.
4. Trả lời các câu hỏi audit: cùng instance set? đủ 10 seed? cùng iteration
   budget? có run nào archive rỗng/infeasible?


In [1]:
# ==== CẤU HÌNH ĐƯỜNG DẪN (chỉnh lại cho đúng máy của bạn) ====
import sys, os
sys.path.append(os.path.abspath("."))  # để import evrp_analysis_utils.py cùng thư mục

FULL_DIR  = "benchmark/full"     # thư mục kết quả bản đầy đủ (equity-aware)
NOEQ_DIR  = "benchmark/no-EQ"    # thư mục kết quả bản loại equity guidance
ARTIFACT_DIR = "artifacts"       # nơi lưu các bảng trung gian (csv) giữa các notebook
os.makedirs(ARTIFACT_DIR, exist_ok=True)

import pandas as pd
import numpy as np
import evrp_analysis_utils as utils

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)


## 1.1. Quét cấu trúc thư mục

In [2]:
runs_full = utils.discover_runs(FULL_DIR)
runs_noeq = utils.discover_runs(NOEQ_DIR)

print(f"FULL:  {runs_full['Instance'].nunique()} instances, {len(runs_full)} runs (instance x seed)")
print(f"No-EQ: {runs_noeq['Instance'].nunique()} instances, {len(runs_noeq)} runs (instance x seed)")

runs_full.head()


FULL:  93 instances, 930 runs (instance x seed)
No-EQ: 93 instances, 930 runs (instance x seed)


,Instance,Seed,RunDir,Prefix
0,c101_21,1,benchmark\full\c101_21\seed_1,c101_21_seed_1
1,c101_21,10,benchmark\full\c101_21\seed_10,c101_21_seed_10
2,c101_21,2,benchmark\full\c101_21\seed_2,c101_21_seed_2
3,c101_21,3,benchmark\full\c101_21\seed_3,c101_21_seed_3
4,c101_21,4,benchmark\full\c101_21\seed_4,c101_21_seed_4


## 1.2. Kiểm tra instance set trùng khớp giữa hai variant

In [3]:
inst_full = set(runs_full["Instance"].unique())
inst_noeq = set(runs_noeq["Instance"].unique())

only_full = inst_full - inst_noeq
only_noeq = inst_noeq - inst_full
matched = inst_full & inst_noeq

print(f"Instances chỉ có ở FULL:  {len(only_full)} -> {sorted(only_full)[:10]}")
print(f"Instances chỉ có ở No-EQ: {len(only_noeq)} -> {sorted(only_noeq)[:10]}")
print(f"Instances matched cả hai: {len(matched)}")


Instances chỉ có ở FULL:  0 -> []
Instances chỉ có ở No-EQ: 0 -> []
Instances matched cả hai: 93


## 1.3. Kiểm tra số seed / instance

In [4]:
seed_counts = (
    pd.concat([
        runs_full.assign(Variant="FULL"),
        runs_noeq.assign(Variant="No-EQ"),
    ])
    .groupby(["Variant", "Instance"])["Seed"].nunique()
    .reset_index(name="NumSeeds")
)

incomplete = seed_counts[seed_counts["NumSeeds"] < 10]
print(f"Số (variant, instance) có ÍT HƠN 10 seed: {len(incomplete)}")
incomplete.sort_values(["Variant", "NumSeeds"]).head(20)


Số (variant, instance) có ÍT HƠN 10 seed: 0


,Variant,Instance,NumSeeds


## 1.4. Xây run_manifest.csv

In [5]:
manifest = utils.build_run_manifest(FULL_DIR, NOEQ_DIR)
manifest.to_csv(f"{ARTIFACT_DIR}/run_manifest.csv", index=False)
print(f"Tổng số dòng manifest: {len(manifest)}")
print(f"Số dòng Valid=True: {manifest['Valid'].sum() if not manifest.empty else 0}")
manifest.head(10)


Tổng số dòng manifest: 1860
Số dòng Valid=True: 1840


,Variant,Instance,Seed,RunDir,ConfigHash,Runtime_s,Iterations,FinalArchiveSize_reported,FrontObjRows,HasFrontObjectives,HasFrontDetails,HasEvolutionLog,HasProgress,HasOperators,Valid,Group,Size
0,FULL,c101_21,1,benchmark\full\c101_21\seed_1,d8f9b899d6,775.317,25000.0,30.0,30,True,True,True,True,True,True,C,21.0
1,FULL,c101_21,10,benchmark\full\c101_21\seed_10,bbac9020ed,737.186,25000.0,32.0,32,True,True,True,True,True,True,C,21.0
2,FULL,c101_21,2,benchmark\full\c101_21\seed_2,6a0f35a2bd,800.131,25000.0,43.0,43,True,True,True,True,True,True,C,21.0
3,FULL,c101_21,3,benchmark\full\c101_21\seed_3,29f6a8f112,747.130,25000.0,28.0,28,True,True,True,True,True,True,C,21.0
4,FULL,c101_21,4,benchmark\full\c101_21\seed_4,0dfca08bd7,778.205,25000.0,28.0,28,True,True,True,True,True,True,C,21.0
5,FULL,c101_21,5,benchmark\full\c101_21\seed_5,17b2d446cc,634.036,25000.0,48.0,48,True,True,True,True,True,True,C,21.0
6,FULL,c101_21,6,benchmark\full\c101_21\seed_6,9119abb399,887.029,25000.0,37.0,37,True,True,True,True,True,True,C,21.0
7,FULL,c101_21,7,benchmark\full\c101_21\seed_7,0ac50b01e0,996.071,25000.0,32.0,32,True,True,True,True,True,True,C,21.0
8,FULL,c101_21,8,benchmark\full\c101_21\seed_8,bbb682919c,866.742,25000.0,38.0,38,True,True,True,True,True,True,C,21.0
9,FULL,c101_21,9,benchmark\full\c101_21\seed_9,a3a053e041,1020.037,25000.0,47.0,47,True,True,True,True,True,True,C,21.0


## 1.5. Kiểm tra ConfigHash — cùng cấu hình thuật toán không?

Nếu một variant có nhiều ConfigHash khác nhau *trong cùng variant*, đó là dấu hiệu
chạy không đồng nhất (khác maxIterations, khác tham số...) cần xử lý trước khi so sánh.

In [6]:
hash_summary = manifest.groupby("Variant")["ConfigHash"].nunique()
print(hash_summary)

if not manifest.empty:
    for variant, g in manifest.groupby("Variant"):
        n_hash = g["ConfigHash"].nunique()
        if n_hash > 1:
            print(f"\n[CẢNH BÁO] Variant {variant} có {n_hash} ConfigHash khác nhau:")
            print(g["ConfigHash"].value_counts())


Variant
FULL     930
No-EQ    930
Name: ConfigHash, dtype: int64

[CẢNH BÁO] Variant FULL có 930 ConfigHash khác nhau:
ConfigHash
b14b1806f3    1
d8f9b899d6    1
dc9982f41e    1
63d8cef427    1
5268f6cd93    1
             ..
9119abb399    1
17b2d446cc    1
0dfca08bd7    1
29f6a8f112    1
6a0f35a2bd    1
Name: count, Length: 930, dtype: int64

[CẢNH BÁO] Variant No-EQ có 930 ConfigHash khác nhau:
ConfigHash
50340832c2    1
599c447b00    1
9e4ff4336c    1
88238cb79f    1
962d8725be    1
             ..
9556c13460    1
0577a29ad8    1
e150877bff    1
ee46a0a6ed    1
80eb839f8e    1
Name: count, Length: 930, dtype: int64


## 1.6. Kiểm tra iteration budget đồng nhất giữa FULL và No-EQ

Theo tài liệu: *'All variants use a common iteration budget, not a common wall-clock-time budget'*.
Cell dưới kiểm tra giả định này và báo cáo runtime overhead của FULL so với No-EQ.

In [7]:
iter_by_variant = manifest.groupby("Variant")["Iterations"].agg(["mean", "std", "min", "max"])
runtime_by_variant = manifest.groupby("Variant")["Runtime_s"].agg(["mean", "std", "min", "max"])

print("=== Iterations ===")
print(iter_by_variant)
print("\n=== Runtime (s) ===")
print(runtime_by_variant)

if set(["FULL", "No-EQ"]).issubset(iter_by_variant.index):
    overhead = (runtime_by_variant.loc["FULL", "mean"] / runtime_by_variant.loc["No-EQ", "mean"] - 1) * 100
    print(f"\nFULL runtime overhead trung bình so với No-EQ: {overhead:.1f}%")


=== Iterations ===
            mean  std      min      max
Variant                                
FULL     25000.0  0.0  25000.0  25000.0
No-EQ    25000.0  0.0  25000.0  25000.0

=== Runtime (s) ===
               mean         std    min       max
Variant                                         
FULL     364.936689  345.368740  0.030  1450.476
No-EQ    317.230379  326.566138  0.029  1485.275

FULL runtime overhead trung bình so với No-EQ: 15.0%


## 1.7. Runs cần loại khỏi main analysis (archive rỗng / thiếu file)

In [8]:
bad_runs = manifest[~manifest["Valid"]] if not manifest.empty else manifest
print(f"Số run KHÔNG hợp lệ (sẽ loại khỏi main analysis): {len(bad_runs)}")
bad_runs[["Variant", "Instance", "Seed", "HasFrontObjectives", "FrontObjRows", "HasProgress"]].head(30)


Số run KHÔNG hợp lệ (sẽ loại khỏi main analysis): 20


,Variant,Instance,Seed,HasFrontObjectives,FrontObjRows,HasProgress
290,FULL,output1,1,True,0,True
291,FULL,output1,10,True,0,True
292,FULL,output1,2,True,0,True
293,FULL,output1,3,True,0,True
294,FULL,output1,4,True,0,True
295,FULL,output1,5,True,0,True
296,FULL,output1,6,True,0,True
297,FULL,output1,7,True,0,True
298,FULL,output1,8,True,0,True
299,FULL,output1,9,True,0,True


## 1.8. Quyết định sample rule (Section 2 của tài liệu)

So sánh: dùng **toàn bộ instance matched** làm main experiment, và subgroup theo size
(vd. `_21` = 100-customer) làm robustness check, thay vì chọn subgroup sau khi đã thấy kết quả.

In [9]:
manifest_matched = manifest[manifest["Instance"].isin(matched) & manifest["Valid"]]
print(f"Số run trong main analysis (matched + valid): {len(manifest_matched)}")

size_counts = manifest_matched.groupby("Size")["Instance"].nunique()
print("\nSố instance theo Size (kiểm tra subgroup 100-customer):")
print(size_counts)

manifest_matched.to_csv(f"{ARTIFACT_DIR}/run_manifest_matched_valid.csv", index=False)
print(f"\nĐã lưu: {ARTIFACT_DIR}/run_manifest_matched_valid.csv")
print("--> Notebook 02 sẽ dùng file này làm input.")


Số run trong main analysis (matched + valid): 1840

Số instance theo Size (kiểm tra subgroup 100-customer):
Size
5.0     12
10.0    12
15.0    12
21.0    56
Name: Instance, dtype: int64

Đã lưu: artifacts/run_manifest_matched_valid.csv
--> Notebook 02 sẽ dùng file này làm input.
